# Imports and Data Load

In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
import spacy
import re
import nltk
import numpy as np
nltk.download("stopwords")
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from textblob import TextBlob
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.decomposition import TruncatedSVD
from sklearn.naive_bayes import MultinomialNB

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [48]:
train_data = pd.read_csv('https://drive.google.com/uc?id=1-fZUIY8HJ_P6M8bQ2LUITwR-G_zgIDK5')
test_data = pd.read_csv('https://drive.google.com/uc?id=1k-mFYjbegv0OW5NgF1f0gMX8bAQgSL6V')

In [49]:
X_train_full = train_data.drop(columns=['Sentiment'])
y_train_full = train_data['Sentiment']
X_test = test_data

X_train, X_val, y_train, y_val = train_test_split(
    train_data['Text'],
    train_data['Sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=train_data['Sentiment']
)

# Preprocess

In [50]:
class Preprocess(BaseEstimator, TransformerMixin):
    def __init__(self,
                 lowercase=True,
                 spelling_correction=False,
                 lemmatize=True,
                 pos_filter=False,
                 entity_masking=False):
        self.lowercase = lowercase
        self.spelling_correction = spelling_correction
        self.lemmatize = lemmatize
        self.pos_filter = pos_filter
        self.entity_masking = entity_masking
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()
        self.spacy_nlp = spacy.load("en_core_web_sm")

    def correct_spelling(self, text):
        return str(TextBlob(text).correct())

    def mask_entities(self, text):
        doc = self.spacy_nlp(text)
        for ent in doc.ents:
            text = text.replace(ent.text, f"<{ent.label_}>")
        return text

    def preprocess_text(self, text):
        # Lowercase
        if self.lowercase:
            text = text.lower()

        # Mask entities
        if self.entity_masking:
            text = self.mask_entities(text)

        # Remove unwanted characters
        text = re.sub(r"[^a-z\s]", " ", text)

        # Spelling correction
        if self.spelling_correction:
            text = self.correct_spelling(text)

        tokens = text.split()

        # POS Tagger filtering
        if self.pos_filter:
            doc = self.spacy_nlp(" ".join(tokens))
            tokens = [token.text for token in doc if token.pos_ in ["ADJ", "ADV", "VERB", "NOUN"]]

        # Stopword elimination
        tokens = [w for w in tokens if w not in self.stop_words]

        # Lemmatization
        if self.lemmatize:
            tokens = [self.lemmatizer.lemmatize(w) for w in tokens]

        return " ".join(tokens)

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return [self.preprocess_text(text) for text in X]

# Feature Extraction

In [51]:
features = FeatureUnion([
    ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)),
    ("bow", CountVectorizer(max_features=1000, ngram_range=(1,1))),
])

# Machine Learning Algorithm

### Logistic Regression

In [52]:
log_reg = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False, # Sebaiknya jangan dihidupkan wkwk (lambat kali wak)
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("svd", TruncatedSVD(n_components=500, random_state=42)),
    ("clf", LogisticRegression(C=3, max_iter=1000, solver="liblinear"))
])

### Train & Evaluate Logistic Regression

In [53]:
log_reg.fit(X_train, y_train)
y_val_pred_log_reg = log_reg.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_log_reg))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_log_reg))

Validation Accuracy: 0.8898305084745762

Classification Report:
               precision    recall  f1-score   support

    Negative       0.91      0.96      0.94        99
    Positive       0.71      0.53      0.61        19

    accuracy                           0.89       118
   macro avg       0.81      0.74      0.77       118
weighted avg       0.88      0.89      0.88       118



### Linear SVC

In [54]:
svc = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False,
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("svd", TruncatedSVD(n_components=500, random_state=42)),
    ("clf", LinearSVC(C=1, max_iter=2000))
])

### Train & Evaluate Linear SVC

In [55]:
svc.fit(X_train, y_train)
y_val_pred_svc = svc.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_svc))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_svc))

Validation Accuracy: 0.9067796610169492

Classification Report:
               precision    recall  f1-score   support

    Negative       0.94      0.95      0.94        99
    Positive       0.72      0.68      0.70        19

    accuracy                           0.91       118
   macro avg       0.83      0.82      0.82       118
weighted avg       0.90      0.91      0.91       118



### Model Multinomial NB

In [56]:
multi_nb = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False,
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("clf", MultinomialNB(alpha=1.0))
])

### Train & Evaluate Multinomial NB

In [57]:
multi_nb.fit(X_train, y_train)
y_val_pred_nb = multi_nb.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_nb))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_nb))

Validation Accuracy: 0.8728813559322034

Classification Report:
               precision    recall  f1-score   support

    Negative       0.89      0.97      0.93        99
    Positive       0.70      0.37      0.48        19

    accuracy                           0.87       118
   macro avg       0.79      0.67      0.71       118
weighted avg       0.86      0.87      0.86       118



# Predict

### Predict Logistic Regression

In [58]:
test_predictions_log_reg = log_reg.predict(X_test["Text"])
submission_df_log_reg = pd.DataFrame({"id": X_test["id"], "Sentiment": test_predictions_log_reg})
submission_df_log_reg.to_csv("logreg.csv", index=False)

### Predict Linear SVC

In [62]:
test_predictions_linear_svc = svc.predict(X_test["Text"])
submission_df_linear_svc = pd.DataFrame({"id": X_test["id"], "Sentiment": test_predictions_linear_svc})
submission_df_linear_svc.to_csv("lin_svc.csv", index=False)

### Predict Multinomial NB

In [63]:
test_predictions_multi_nb = multi_nb.predict(X_test["Text"])
submission_df_multi_nb = pd.DataFrame({"id": X_test["id"], "Sentiment": test_predictions_multi_nb})
submission_df_multi_nb.to_csv("multi_nb.csv", index=False)

# Feature Extraction Analysis

### Bag of Words

Without BoW

In [68]:
features = FeatureUnion([
    ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)),
])

log_reg_no_bow = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False,
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("clf", LogisticRegression(C=3, max_iter=1000, solver="liblinear"))
])

log_reg_no_bow.fit(X_train, y_train)
y_val_pred_log_reg_no_bow = log_reg.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_log_reg_no_bow))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_log_reg_no_bow))

Validation Accuracy: 0.8898305084745762

Classification Report:
               precision    recall  f1-score   support

    Negative       0.91      0.96      0.94        99
    Positive       0.71      0.53      0.61        19

    accuracy                           0.89       118
   macro avg       0.81      0.74      0.77       118
weighted avg       0.88      0.89      0.88       118



With BoW

In [69]:
features = FeatureUnion([
    ("bow", CountVectorizer(max_features=1000, ngram_range=(1,1))),
    ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)),
])

log_reg_with_bow = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False,
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("clf", LogisticRegression(C=3, max_iter=1000, solver="liblinear"))
])

log_reg_with_bow.fit(X_train, y_train)
y_val_pred_log_reg_with_bow = log_reg_with_bow.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_log_reg_with_bow))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_log_reg_with_bow))

Validation Accuracy: 0.8898305084745762

Classification Report:
               precision    recall  f1-score   support

    Negative       0.91      0.96      0.94        99
    Positive       0.71      0.53      0.61        19

    accuracy                           0.89       118
   macro avg       0.81      0.74      0.77       118
weighted avg       0.88      0.89      0.88       118



### TF-IDF

Without TF-IDF

In [71]:
features = FeatureUnion([
    ("bow", CountVectorizer(max_features=1000, ngram_range=(1,1))),
])

log_reg_no_tf_idf = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False,
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("clf", LogisticRegression(C=3, max_iter=1000, solver="liblinear"))
])

log_reg_no_tf_idf.fit(X_train, y_train)
y_val_pred_log_reg_no_tf_idf = log_reg.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_log_reg_no_tf_idf))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_log_reg_no_tf_idf))

Validation Accuracy: 0.8898305084745762

Classification Report:
               precision    recall  f1-score   support

    Negative       0.91      0.96      0.94        99
    Positive       0.71      0.53      0.61        19

    accuracy                           0.89       118
   macro avg       0.81      0.74      0.77       118
weighted avg       0.88      0.89      0.88       118



With TF-IDF

In [72]:
features = FeatureUnion([
    ("bow", CountVectorizer(max_features=1000, ngram_range=(1,1))),
    ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)),
])

log_reg_with_tf_idf = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False,
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("clf", LogisticRegression(C=3, max_iter=1000, solver="liblinear"))
])

log_reg_with_tf_idf.fit(X_train, y_train)
y_val_pred_log_reg_with_tf_idf = log_reg_with_tf_idf.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_log_reg_with_tf_idf))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_log_reg_with_tf_idf))

Validation Accuracy: 0.8898305084745762

Classification Report:
               precision    recall  f1-score   support

    Negative       0.91      0.96      0.94        99
    Positive       0.71      0.53      0.61        19

    accuracy                           0.89       118
   macro avg       0.81      0.74      0.77       118
weighted avg       0.88      0.89      0.88       118



### Dimensionality Reduction (SVD)

Without SVD

In [75]:
features = FeatureUnion([
    ("bow", CountVectorizer(max_features=1000, ngram_range=(1,1))),
    ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)),
])

log_reg_without_svd = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False,
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("clf", LogisticRegression(C=3, max_iter=1000, solver="liblinear"))
])

log_reg_without_svd.fit(X_train, y_train)
y_val_pred_log_reg_without_svd = log_reg_without_svd.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_log_reg_without_svd))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_log_reg_without_svd))

Validation Accuracy: 0.8898305084745762

Classification Report:
               precision    recall  f1-score   support

    Negative       0.91      0.96      0.94        99
    Positive       0.71      0.53      0.61        19

    accuracy                           0.89       118
   macro avg       0.81      0.74      0.77       118
weighted avg       0.88      0.89      0.88       118



With SVD

In [76]:
features = FeatureUnion([
    ("bow", CountVectorizer(max_features=1000, ngram_range=(1,1))),
    ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)),
])

log_reg_with_svd = Pipeline([
    ("preprocess", Preprocess(
        lowercase=True,
        spelling_correction=False,
        lemmatize=True,
        pos_filter=True,
        entity_masking=True,
    )),
    ("features", features),
    ("svd", TruncatedSVD(n_components=500, random_state=42)),
    ("clf", LogisticRegression(C=3, max_iter=1000, solver="liblinear"))
])

log_reg_with_svd.fit(X_train, y_train)
y_val_pred_log_reg_with_svd = log_reg_with_svd.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_log_reg_with_svd))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_log_reg_with_svd))

Validation Accuracy: 0.8898305084745762

Classification Report:
               precision    recall  f1-score   support

    Negative       0.91      0.96      0.94        99
    Positive       0.71      0.53      0.61        19

    accuracy                           0.89       118
   macro avg       0.81      0.74      0.77       118
weighted avg       0.88      0.89      0.88       118



# Others

### Find best parameter for TF-IDF and BoW

In [64]:
from sklearn.model_selection import GridSearchCV

params = {
    "features__tfidf__max_features": [2000, 5000, 10000, 20000],
    "features__bow__max_features": [1000, 3000, 5000]
}

grid = GridSearchCV(log_reg, param_grid=params, cv=5, scoring="f1_macro")
grid.fit(X_train, y_train)
print(grid.best_params_)

{'features__bow__max_features': 1000, 'features__tfidf__max_features': 2000}


### Find best parameter for Linear SVC

In [66]:
from sklearn.model_selection import GridSearchCV

params = {
    "clf__C": [0.01, 0.1, 1, 3, 10]
}

grid = GridSearchCV(
    svc,
    param_grid=params,
    cv=5,
    scoring="f1_macro",
    n_jobs=1
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best score:", grid.best_score_)

Best params: {'clf__C': 1}
Best score: 0.7752766637737641
